In [1]:
# ============================================================
# NOTEBOOK 1: DATASET PREPARATION
# Run this once — saves all datasets to /kaggle/working/datasets/
# ============================================================

import os
import json
import random
import pandas as pd
import numpy as np
import requests
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Directory setup
base_path = '/kaggle/working/datasets'
os.makedirs(base_path, exist_ok=True)
os.makedirs(f'{base_path}/alpaca', exist_ok=True)
os.makedirs(f'{base_path}/oa', exist_ok=True)

print("="*60)
print("DATASET PREPARATION")
print("="*60)

# ============================================================
# 1. PILE DATASET (Streaming, 50k samples)
# ============================================================
print("\n[1/5] Loading PILE dataset (streaming)...")

pile_save_path = f'{base_path}/pile_50k.csv'

if os.path.exists(pile_save_path):
    print(f"   ✅ Already exists: {pile_save_path}")
    pile_df = pd.read_csv(pile_save_path)
    print(f"   Samples: {len(pile_df)}")
else:
    parts = [
        {'skip': 0,      'target': 17000, 'label': 'Start'},
        {'skip': 200000, 'target': 17000, 'label': 'Middle'},
        {'skip': 500000, 'target': 16000, 'label': 'End'},
    ]

    all_samples = []

    for part in parts:
        print(f"\n   📥 Collecting from {part['label']} (skip={part['skip']})...")

        dataset = load_dataset(
            "monology/pile-uncopyrighted",
            split="train",
            streaming=True
        )

        samples = []
        skipped = 0
        processed = 0

        for example in dataset:
            if skipped < part['skip']:
                skipped += 1
                continue

            text = example.get('text', '').strip()
            if len(text) > 100:
                samples.append({
                    'text': text,
                    'source': example.get('meta', {}).get('pile_set_name', 'unknown')
                })
                processed += 1

            if processed >= part['target']:
                break

            if processed % 5000 == 0 and processed > 0:
                print(f"      Collected {processed}/{part['target']}...")

        print(f"   ✅ Collected {len(samples)} from {part['label']}")
        all_samples.extend(samples)

    print(f"\n   🔀 Shuffling {len(all_samples)} samples...")
    random.shuffle(all_samples)

    pile_df = pd.DataFrame(all_samples[:50000])
    pile_df.to_csv(pile_save_path, index=False)
    print(f"   ✅ PILE saved: {len(pile_df)} samples → {pile_save_path}")
    print(f"   Source distribution:\n{pile_df['source'].value_counts().head(5)}")

# ============================================================
# 2. T-REx DATASET (KILT, streaming download)
# ============================================================
print("\n[2/5] Loading T-REx (KILT) dataset...")

trex_save_path = f'{base_path}/trex_50k.csv'

if os.path.exists(trex_save_path):
    print(f"   ✅ Already exists: {trex_save_path}")
    trex_df = pd.read_csv(trex_save_path)
    print(f"   Samples: {len(trex_df)}")
else:
    url = "http://dl.fbaipublicfiles.com/KILT/trex-train-kilt.jsonl"
    print(f"   Downloading from: {url}")
    response = requests.get(url, stream=True)

    samples = []
    target = 50000
    raw_lines = 0

    for line in response.iter_lines():
        if len(samples) >= target:
            break
        if not line:
            continue

        raw_lines += 1
        try:
            item = json.loads(line)
            input_text = item.get('input', '').strip()
            outputs = item.get('output', [])
            if not outputs:
                continue
            answer = outputs[0].get('answer', '').strip()
            if not answer:
                continue
            if '[SEP]' not in input_text:
                continue

            parts = input_text.split('[SEP]')
            subject = parts[0].strip()
            relation = parts[1].strip() if len(parts) > 1 else ''
            text = f"{subject}'s {relation} is"
            entity = answer

            samples.append({
                'text': text,
                'entity': entity,
                'subject': subject,
                'relation': relation
            })

        except Exception:
            continue

        if raw_lines % 100000 == 0:
            print(f"   Processed {raw_lines} lines, collected {len(samples)}...")

    response.close()
    trex_df = pd.DataFrame(samples[:target])
    trex_df.to_csv(trex_save_path, index=False)
    print(f"   ✅ T-REx saved: {len(trex_df)} samples → {trex_save_path}")

# ============================================================
# 3. MMLU DATASET
# ============================================================
print("\n[3/5] Loading MMLU dataset...")

mmlu_test_path = f'{base_path}/mmlu_test.csv'
mmlu_val_path  = f'{base_path}/mmlu_val.csv'

if os.path.exists(mmlu_test_path) and os.path.exists(mmlu_val_path):
    print(f"   ✅ Already exists")
    mmlu_df     = pd.read_csv(mmlu_test_path)
    mmlu_val_df = pd.read_csv(mmlu_val_path)
    print(f"   Test samples:  {len(mmlu_df)}")
    print(f"   Val samples:   {len(mmlu_val_df)}")
else:
    mmlu     = load_dataset("cais/mmlu", "all", split="test")
    mmlu_val = load_dataset("cais/mmlu", "all", split="validation")

    mmlu_df     = pd.DataFrame(mmlu)
    mmlu_val_df = pd.DataFrame(mmlu_val)

    mmlu_df.to_csv(mmlu_test_path, index=False)
    mmlu_val_df.to_csv(mmlu_val_path, index=False)
    print(f"   ✅ MMLU test saved:  {len(mmlu_df)} samples")
    print(f"   ✅ MMLU val saved:   {len(mmlu_val_df)} samples")
    print(f"   Subjects: {mmlu_df['subject'].nunique()}")

# ============================================================
# 4. ALPACA DATASET (Fine-tuning data)
# ============================================================
print("\n[4/5] Loading Alpaca dataset...")

alpaca_path = f'{base_path}/alpaca/alpaca_data.csv'

if os.path.exists(alpaca_path):
    print(f"   ✅ Already exists")
    alpaca_df = pd.read_csv(alpaca_path)
    print(f"   Samples: {len(alpaca_df)}")
else:
    alpaca_raw = load_dataset("tatsu-lab/alpaca", split="train")

    def format_alpaca(example):
        if example["input"] and example["input"].strip():
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Input:\n{example['input']}\n"
                f"### Response:\n{example['output']}"
            )
        else:
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Response:\n{example['output']}"
            )
        return {"text": text}

    alpaca_formatted = alpaca_raw.map(format_alpaca)
    alpaca_df = pd.DataFrame({
        'text': alpaca_formatted['text'],
        'instruction': alpaca_formatted['instruction'],
        'input': alpaca_formatted['input'],
        'output': alpaca_formatted['output'],
    })
    alpaca_df.to_csv(alpaca_path, index=False)
    print(f"   ✅ Alpaca saved: {len(alpaca_df)} samples → {alpaca_path}")

# ============================================================
# 5. OPENASSISTANT (OA) DATASET (Fine-tuning data)
# ============================================================
print("\n[5/5] Loading OpenAssistant (OA) dataset...")

oa_path = f'{base_path}/oa/oa_data.csv'

if os.path.exists(oa_path):
    print(f"   ✅ Already exists")
    oa_df = pd.read_csv(oa_path)
    print(f"   Samples: {len(oa_df)}")
else:
    oa_raw = load_dataset("OpenAssistant/oasst1", split="train")

    # Paper: extract single-turn English conversations
    oa_samples = []
    for example in oa_raw:
        if (example.get('lang') == 'en' and
            example.get('role') == 'prompter' and
            example.get('text')):

            text = (
                f"### Instruction:\n{example['text']}\n"
                f"### Response:\n"
            )
            oa_samples.append({
                'text': text,
                'raw_text': example['text'],
                'message_id': example.get('message_id', ''),
            })

    oa_df = pd.DataFrame(oa_samples)
    # Paper uses 11k pairs — sample down
    oa_df = oa_df.sample(n=min(11000, len(oa_df)), random_state=SEED).reset_index(drop=True)
    oa_df.to_csv(oa_path, index=False)
    print(f"   ✅ OA saved: {len(oa_df)} samples → {oa_path}")

# For fair comparison, sample Alpaca to same size as OA (paper does this)
alpaca_sampled_path = f'{base_path}/alpaca/alpaca_sampled.csv'
if not os.path.exists(alpaca_sampled_path):
    alpaca_sampled = alpaca_df.sample(
        n=min(11000, len(alpaca_df)),
        random_state=SEED
    ).reset_index(drop=True)
    alpaca_sampled.to_csv(alpaca_sampled_path, index=False)
    print(f"   ✅ Alpaca sampled to {len(alpaca_sampled)} (matching OA size)")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("ALL DATASETS READY")
print("="*60)
print(f"  PILE       : {pile_save_path}")
print(f"  T-REx      : {trex_save_path}")
print(f"  MMLU test  : {mmlu_test_path}")
print(f"  MMLU val   : {mmlu_val_path}")
print(f"  Alpaca     : {alpaca_path}")
print(f"  Alpaca(11k): {alpaca_sampled_path}")
print(f"  OA         : {oa_path}")

DATASET PREPARATION

[1/5] Loading PILE dataset (streaming)...

   📥 Collecting from Start (skip=0)...


README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

      Collected 5000/17000...
      Collected 10000/17000...
      Collected 15000/17000...
   ✅ Collected 17000 from Start

   📥 Collecting from Middle (skip=200000)...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

...

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

   ✅ OA saved: 11000 samples → /kaggle/working/datasets/oa/oa_data.csv
   ✅ Alpaca sampled to 11000 (matching OA size)

ALL DATASETS READY
  PILE       : /kaggle/working/datasets/pile_50k.csv
  T-REx      : /kaggle/working/datasets/trex_50k.csv
  MMLU test  : /kaggle/working/datasets/mmlu_test.csv
  MMLU val   : /kaggle/working/datasets/mmlu_val.csv
  Alpaca     : /kaggle/working/datasets/alpaca/alpaca_data.csv
  Alpaca(11k): /kaggle/working/datasets/alpaca/alpaca_sampled.csv
  OA         : /kaggle/working/datasets/oa/oa_data.csv


In [2]:
# ============================================================
# NOTEBOOK 2 PART 1: Setup + CLM + Facts Generation
# ============================================================

import os, gc, json, ast, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Verify GPUs ──────────────────────────────────────────────
print("GPU Check:")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

# ── Paths & Config ───────────────────────────────────────────
MODEL_PATH   = '/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1'
DATA_PATH    = "/kaggle/working/datasets"
RESULTS_PATH = "/kaggle/working/results"
os.makedirs(RESULTS_PATH, exist_ok=True)

SEED         = 42
EVAL_SAMPLES = 5000
N_BINS       = 10

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# ── Load Model ───────────────────────────────────────────────
print("\nLoading LLaMA-7B (fp16) across both T4 GPUs...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print(f"\nModel loaded!")
print(f"Device map (first 5 layers):")
for k, v in list(model.hf_device_map.items())[:5]:
    print(f"  {k}: {v}")
print(f"\nGPU Memory after loading:")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {used:.2f} / {total:.2f} GB used")

# ── ECE Utilities ────────────────────────────────────────────
def compute_ece(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(
            accuracies[mask].mean() - confidences[mask].mean()
        )
    return float(ece)

def compute_reliability_diagram_data(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        bin_sizes.append(int(mask.sum()))
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append(float((bins[i] + bins[i+1]) / 2))
        else:
            bin_accs.append(float(accuracies[mask].mean()))
            bin_confs.append(float(confidences[mask].mean()))
    return bin_accs, bin_confs, bin_sizes

# ── Load Evaluation Datasets ─────────────────────────────────
print("\nLoading evaluation datasets...")
pile_df = pd.read_csv(f'{DATA_PATH}/pile_50k.csv')
trex_df = pd.read_csv(f'{DATA_PATH}/trex_50k.csv')

pile_eval = pile_df.sample(
    n=EVAL_SAMPLES, random_state=SEED
).reset_index(drop=True)

trex_eval = trex_df.sample(
    n=EVAL_SAMPLES, random_state=SEED
).reset_index(drop=True)

print(f"  PILE eval samples : {len(pile_eval)}")
print(f"  T-REx eval samples: {len(trex_eval)}")

# ── TASK 1: CLM (PILE) ───────────────────────────────────────
print("\n" + "="*60)
print("TASK 1: Causal Language Modeling (PILE)")
print("="*60)

# Baseline: no instruction prompt
clm_confidences = []
clm_accuracies  = []

clm_start_time = time.time() # Added timer start
for idx, row in pile_eval.iterrows():
    text = str(row['text']).strip()
    try:
        inputs    = tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=512
        )
        input_ids = inputs["input_ids"]
        seq_len   = input_ids.shape[1]
        if seq_len < 3:
            continue

        pos        = np.random.randint(1, seq_len)
        context    = input_ids[:, :pos].to(model.device)
        true_token = input_ids[:, pos].item()

        with torch.no_grad():
            out   = model(input_ids=context)
            probs = F.softmax(
                out.logits[:, -1, :].float(), dim=-1
            )

        clm_confidences.append(probs.max(dim=-1).values.item())
        clm_accuracies.append(
            int(probs.argmax(dim=-1).item() == true_token)
        )

    except Exception as e:
        continue

    # Changed 5000 to 1000 for faster updates, added percentage and timer
    if (idx + 1) % 1000 == 0:
        elapsed_mins = (time.time() - clm_start_time) / 60
        pct = (idx + 1) / EVAL_SAMPLES * 100
        print(f"  [{idx+1}/{EVAL_SAMPLES}] ({pct:.1f}%) | Time: {elapsed_mins:.1f}m | "
              f"ECE: {compute_ece(clm_confidences, clm_accuracies):.4f} | "
              f"Acc: {np.mean(clm_accuracies):.4f}")

clm_ece = compute_ece(clm_confidences, clm_accuracies)
clm_acc = float(np.mean(clm_accuracies))
clm_bins = compute_reliability_diagram_data(clm_confidences, clm_accuracies)
print(f"\n✅ CLM Done | ECE: {clm_ece:.4f} | Accuracy: {clm_acc:.4f}")

# ── TASK 2: Facts Generation (T-REx) ────────────────────────
print("\n" + "="*60)
print("TASK 2: Facts Generation (T-REx)")
print("="*60)

# Baseline: no instruction prompt
facts_confidences = []
facts_accuracies  = []

facts_start_time = time.time() # Added timer start
for idx, row in trex_eval.iterrows():
    text   = str(row['text']).strip()
    entity = str(row['entity']).strip()

    try:
        entity_tokens = tokenizer(
            entity,
            add_special_tokens=False,
            return_tensors="pt"
        )["input_ids"]

        if entity_tokens.shape[1] == 0:
            continue

        true_first_token = entity_tokens[0, 0].item()

        context_ids = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        )["input_ids"].to(model.device)

        with torch.no_grad():
            out   = model(input_ids=context_ids)
            probs = F.softmax(
                out.logits[:, -1, :].float(), dim=-1
            )

        facts_confidences.append(probs.max(dim=-1).values.item())
        facts_accuracies.append(
            int(probs.argmax(dim=-1).item() == true_first_token)
        )

    except Exception:
        continue

    # Changed 5000 to 1000 for faster updates, added percentage and timer
    if (idx + 1) % 1000 == 0:
        elapsed_mins = (time.time() - facts_start_time) / 60
        pct = (idx + 1) / EVAL_SAMPLES * 100
        print(f"  [{idx+1}/{EVAL_SAMPLES}] ({pct:.1f}%) | Time: {elapsed_mins:.1f}m | "
              f"ECE: {compute_ece(facts_confidences, facts_accuracies):.4f} | "
              f"Acc: {np.mean(facts_accuracies):.4f}")

facts_ece = compute_ece(facts_confidences, facts_accuracies)
facts_acc = float(np.mean(facts_accuracies))
facts_bins = compute_reliability_diagram_data(facts_confidences, facts_accuracies)
print(f"\n✅ Facts Done | ECE: {facts_ece:.4f} | Accuracy: {facts_acc:.4f}")

# ── Save Part 1 Results (intermediate) ───────────────────────
part1_results = {
    "CLM":   {"ECE": clm_ece,   "Accuracy": clm_acc,
               "n_samples": len(clm_confidences),
               "bin_accs": clm_bins[0], "bin_confs": clm_bins[1],
               "bin_sizes": clm_bins[2],
               "confidences": clm_confidences,
               "accuracies":  clm_accuracies},
    "Facts": {"ECE": facts_ece, "Accuracy": facts_acc,
               "n_samples": len(facts_confidences),
               "bin_accs": facts_bins[0], "bin_confs": facts_bins[1],
               "bin_sizes": facts_bins[2],
               "confidences": facts_confidences,
               "accuracies":  facts_accuracies},
}

with open(f'{RESULTS_PATH}/baseline_part1.json', 'w') as f:
    json.dump(part1_results, f, indent=2)

print("\n✅ Part 1 results saved. Now run Part 2 cell.")
print(f"   GPU Memory:")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    print(f"   GPU {i}: {used:.2f} GB used")

GPU Check:
  CUDA available: True
  GPU count: 2
  GPU 0: Tesla T4 | Memory: 15.6 GB
  GPU 1: Tesla T4 | Memory: 15.6 GB

Loading LLaMA-7B (fp16) across both T4 GPUs...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Model loaded!
Device map (first 5 layers):
  model.embed_tokens: 0
  model.layers.0: 0
  model.layers.1: 0
  model.layers.2: 0
  model.layers.3: 0

GPU Memory after loading:
  GPU 0: 6.74 / 15.64 GB used
  GPU 1: 6.74 / 15.64 GB used

Loading evaluation datasets...
  PILE eval samples : 5000
  T-REx eval samples: 5000

TASK 1: Causal Language Modeling (PILE)
  [1000/5000] (20.0%) | Time: 3.3m | ECE: 0.0152 | Acc: 0.6180
  [2000/5000] (40.0%) | Time: 6.6m | ECE: 0.0154 | Acc: 0.6130
  [3000/5000] (60.0%) | Time: 10.0m | ECE: 0.0125 | Acc: 0.6103
  [4000/5000] (80.0%) | Time: 13.5m | ECE: 0.0082 | Acc: 0.6155
  [5000/5000] (100.0%) | Time: 17.0m | ECE: 0.0092 | Acc: 0.6140

✅ CLM Done | ECE: 0.0092 | Accuracy: 0.6140

TASK 2: Facts Generation (T-REx)
  [1000/5000] (20.0%) | Time: 1.1m | ECE: 0.0645 | Acc: 0.1320
  [2000/5000] (40.0%) | Time: 2.2m | ECE: 0.0576 | Acc: 0.1325
  [3000/5000] (60.0%) | Time: 3.3m | ECE: 0.0612 | Acc: 0.1280
  [4000/5000] (80.0%) | Time: 4.4m | ECE: 0.0596 | 

In [3]:
import ast, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import time

# ── Load MMLU ────────────────────────────────────────────────
mmlu_df     = pd.read_csv(f'{DATA_PATH}/mmlu_test.csv')
mmlu_val_df = pd.read_csv(f'{DATA_PATH}/mmlu_val.csv')

def parse_choices(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(x)
    except: return x

mmlu_df['choices']     = mmlu_df['choices'].apply(parse_choices)
mmlu_val_df['choices'] = mmlu_val_df['choices'].apply(parse_choices)

ANSWER_OPTIONS = ['A', 'B', 'C', 'D']
EVAL_MMLU      = 3000
mmlu_eval      = mmlu_df.sample(n=EVAL_MMLU, random_state=42).reset_index(drop=True)
print(f"MMLU eval samples: {len(mmlu_eval)}")

# ── Answer token IDs ─────────────────────────────────────────
answer_token_ids = [
    tokenizer(opt, add_special_tokens=False,
              return_tensors="pt")["input_ids"][0][0].item()
    for opt in ANSWER_OPTIONS
]
print(f"Answer token IDs: {dict(zip(ANSWER_OPTIONS, answer_token_ids))}")

def build_prompt(row):
    subject     = row['subject']
    prompt      = (f"The following are multiple choice questions "
                   f"(with answers) about {subject}.\n\n")
    subject_val = mmlu_val_df[mmlu_val_df['subject'] == subject]
    if len(subject_val) < 5: subject_val = mmlu_val_df
    shots = subject_val.sample(n=min(5, len(subject_val)), random_state=42)
    for _, shot in shots.iterrows():
        ch = shot['choices']
        prompt += f"Question: {shot['question']}\n"
        for i, opt in enumerate(ANSWER_OPTIONS):
            prompt += f"{opt}. {ch[i]}\n"
        prompt += f"Answer: {ANSWER_OPTIONS[int(shot['answer'])]}\n\n"
    ch = row['choices']
    prompt += f"Question: {row['question']}\n"
    for i, opt in enumerate(ANSWER_OPTIONS):
        prompt += f"{opt}. {ch[i]}\n"
    prompt += "Answer:"
    return prompt

# ── MMLU Evaluation ──────────────────────────────────────────
print("\n" + "="*60)
print(f"MMLU ({EVAL_MMLU} samples)...")
print("="*60)
task_start  = time.time()
confs, accs = [], []

for idx, row in mmlu_eval.iterrows():
    try:
        input_ids = tokenizer(
            build_prompt(row), return_tensors="pt",
            truncation=True, max_length=2048
        )["input_ids"].to(model.device)

        with torch.no_grad():
            logits = model(input_ids=input_ids).logits[:, -1, :]

        ans_probs = F.softmax(
            torch.tensor([logits[0, tid].item()
                          for tid in answer_token_ids]).float(),
            dim=-1
        )
        confs.append(ans_probs.max().item())
        accs.append(int(ans_probs.argmax().item() == int(row['answer'])))
    except: continue

    if (idx+1) % 500 == 0:
        elapsed   = time.time() - task_start
        rate      = (idx+1) / elapsed
        remaining = (EVAL_MMLU - (idx+1)) / rate
        print(f"  [{idx+1}/{EVAL_MMLU}] "
              f"ECE: {compute_ece(confs,accs):.4f} | "
              f"Acc: {np.mean(accs):.4f} | "
              f"Elapsed: {elapsed/60:.1f}m | "
              f"ETA: {remaining/60:.1f}m")

mmlu_ece = compute_ece(confs, accs)
mmlu_acc = float(np.mean(accs))
mmlu_bins = compute_reliability_diagram_data(confs, accs)
print(f"\n✅ MMLU ECE: {mmlu_ece:.4f} | Accuracy: {mmlu_acc:.4f} | "
      f"Time: {(time.time()-task_start)/60:.1f}m")

# ── Merge with Part 1 and Save ───────────────────────────────
with open(f'{RESULTS_PATH}/baseline_part1.json', 'r') as f:
    part1 = json.load(f)

baseline_results = {
    "model"       : "LLaMA-7B (baseline, no fine-tuning)",
    "eval_samples": {"CLM": 5000, "Facts": 5000, "MMLU": EVAL_MMLU},
    "CLM"  : part1["CLM"],
    "Facts": part1["Facts"],
    "MMLU" : {
        "ECE"      : mmlu_ece,
        "Accuracy" : mmlu_acc,
        "n_samples": len(confs),
        "bin_accs" : mmlu_bins[0],
        "bin_confs": mmlu_bins[1],
        "bin_sizes": mmlu_bins[2],
    },
}

with open(f'{RESULTS_PATH}/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

os.remove(f'{RESULTS_PATH}/baseline_part1.json')

print(f"\n{'='*60}")
print(f"BASELINE COMPLETE — FINAL SUMMARY")
print(f"{'='*60}")
print(f"  Task         | ECE    | Accuracy")
print(f"  -------------|--------|----------")
print(f"  CLM (PILE)   | {part1['CLM']['ECE']:.4f} | {part1['CLM']['Accuracy']:.4f}")
print(f"  Facts (TREx) | {part1['Facts']['ECE']:.4f} | {part1['Facts']['Accuracy']:.4f}")
print(f"  MMLU         | {mmlu_ece:.4f} | {mmlu_acc:.4f}")
print(f"{'='*60}")
print(f"✅ Saved to: {RESULTS_PATH}/baseline_results.json")

MMLU eval samples: 3000
Answer token IDs: {'A': 319, 'B': 350, 'C': 315, 'D': 360}

MMLU (3000 samples)...
  [500/3000] ECE: 0.0808 | Acc: 0.4380 | Elapsed: 6.1m | ETA: 30.4m
  [1000/3000] ECE: 0.0575 | Acc: 0.4440 | Elapsed: 12.2m | ETA: 24.4m
  [1500/3000] ECE: 0.0586 | Acc: 0.4373 | Elapsed: 18.0m | ETA: 18.0m
  [2000/3000] ECE: 0.0578 | Acc: 0.4395 | Elapsed: 24.1m | ETA: 12.0m
  [2500/3000] ECE: 0.0558 | Acc: 0.4384 | Elapsed: 30.2m | ETA: 6.0m
  [3000/3000] ECE: 0.0497 | Acc: 0.4460 | Elapsed: 36.4m | ETA: 0.0m

✅ MMLU ECE: 0.0497 | Accuracy: 0.4460 | Time: 36.4m

BASELINE COMPLETE — FINAL SUMMARY
  Task         | ECE    | Accuracy
  -------------|--------|----------
  CLM (PILE)   | 0.0092 | 0.6140
  Facts (TREx) | 0.0602 | 0.1288
  MMLU         | 0.0497 | 0.4460
✅ Saved to: /kaggle/working/results/baseline_results.json
